<a href="https://colab.research.google.com/github/blancavazquez/Taller_CienciaDatos_EducacionContinua/blob/main/notebooks/Practica_Streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.2.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.0 pyspark-shell'

In [ ]:
!pip install tensorflow-io
!pip install kafka-python

In [ ]:
#Import packages
import os
from datetime import datetime
import time
import threading
import json
from kafka import KafkaProducer
from kafka.errors import KafkaError
#from sklearn.model_selection import train_test_split
import pandas as pd
#import tensorflow as tf
#import tensorflow_io as tfio

In [ ]:
!wget https://dlcdn.apache.org/kafka/3.8.0/kafka_2.12-3.8.0.tgz

In [ ]:
!ls -ltr ./

In [ ]:
#descomprimir carpeta
!tar -xzf kafka_2.12-3.8.0.tgz

In [ ]:
#Revisando carpeta
!ls -ltr ./

In [ ]:
!cd kafka_2.12-3.8.0
!ls -ltr ./

In [ ]:
!/content/kafka_2.12-3.8.0/bin/zookeeper-server-start.sh -daemon /content/kafka_2.12-3.8.0/config/zookeeper.properties
!/content/kafka_2.12-3.8.0/bin/kafka-server-start.sh -daemon /content/kafka_2.12-3.8.0/config/server.properties
!echo "Waiting for 10 secs until kafka and zookeeper services are up and running"
!sleep 10

In [ ]:
!ps -ef | grep kafka

In [ ]:
!pip install kafka-python

In [ ]:
!/content/kafka_2.12-3.8.0/bin/kafka-topics.sh --create --bootstrap-server 127.0.0.1:9092 --replication-factor 1 --partitions 1 --topic proyecto

In [ ]:
!pip install nltk

------------------

In [ ]:
import time
import json
import requests
from kafka import KafkaProducer

In [ ]:
!pip install praw

In [ ]:
from kafka import KafkaProducer
import json
import praw

# Credenciales que se enviarán a la API de Reddit.
reddit = praw.Reddit(
    client_id="xxxx",
    client_secret="xxxx",
    user_agent="macos:comments-scraper:v1 (by u/Constant_Cupcake3985)",
    check_for_async=False
)

# Subreddit que se desea monitorear
subreddit = reddit.subreddit("worldnews")

In [ ]:
# Instanciar productor
producer = KafkaProducer(bootstrap_servers="localhost:9092")
data_to_download = []

# Iniciar el streaming de comentarios
print(f"Escuchando comentarios en r/{subreddit.display_name} en tiempo real...\n")
for comment in subreddit.stream.comments(skip_existing=True):
    print(f"Nuevo comentario de {comment.author}:\n{comment.body}\n{'-'*40}")
    with open("reddit_comments.txt", "a") as fd:
        fd.write(comment.body.replace('\n', '\\n').replace('\t', '\\t'))
        fd.write("\n")

    # Enviar par autor-comentario (serializado como JSON) al servidor Kafka
    producer.send(topic="proyecto", value=json.dumps([comment.author.name, comment.body]).encode("UTF-8"))

# Consumiendo datos

In [ ]:
#Import packages
import os
import time
import json
import pyspark
import requests
import threading
import pandas as pd
from datetime import datetime
from kafka import KafkaProducer
from kafka.errors import KafkaError

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType
from pyspark.sql.functions import from_json, col,lower, regexp_replace, split, array_join

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('clima').getOrCreate()

In [ ]:
# Consumir mensajes (pares autor-comentario) en tiempo real del tópico
# "reddit" en el servidor Kafka
pair_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "127.0.0.1:9092") \
    .option("subscribe", "proyecto") \
    .option("startingOffsets", "earliest") \
    .load()

In [ ]:
# Convertir los mensajes almacenados en la columna "value" a cadenas (Kafka los almacena en formato binario)
pair_df_string = pair_df.selectExpr("CAST(value AS STRING)")

# Deserializar (convertir de JSON a lista con 2 cadenas: [autor, comentario]) y crear columnas para cada tipo de cadena
schema = ArrayType(StringType())
parsed_df = pair_df_string.withColumn("parsed", from_json("value", schema)).select(col("parsed")[0].alias("author"),col("parsed")[1].alias("comment"))

# Separar las columnas en DataFrames independientes
author_df = parsed_df.select("author")
comment_df = parsed_df.select("comment")

In [ ]:
debug_query = comment_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

In [ ]:
processed_comment_df = comment_df.withColumn("lower", lower(col("comment"))).select("lower")

In [ ]:
!pip install nltk

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType, StructType, StructField, IntegerType, FloatType
from pyspark.sql.functions import from_json, col, lower, regexp_replace, split, array_join
from pyspark.sql import DataFrame, Row
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
english_stopwords = set(stopwords.words('english'))

# Descargar los recursos de NLTK en todos los executors
def setup_nltk():
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    nltk.download('wordnet', quiet=True)
spark.sparkContext.broadcast(setup_nltk())

# *** Funciones auxiliares ***
# Remover palabras cortas (menores a 4 caracteres de longitud) y stopwords
stopwords_list = stopwords.words("english")
stopwords_list += ["like", "make", "think", "want", "say", "talk", "think",
                   "need", "look," "tell", "thing"]
def filter_words(words):
    return [word for word in words if (len(word) > 3) and (word not in stopwords_list)]
filter_words_udf = udf(filter_words, ArrayType(StringType()))

# Lematiza una oración
def lemmatize_sentence(sentence):
    from nltk import word_tokenize, pos_tag
    from nltk.corpus import wordnet
    from nltk.stem import WordNetLemmatizer

    # Convertir las etiquetas POS de NLTK a etiquetas compatibles con WordNet
    def get_wordnet_pos(tag):
        if tag.startswith('J'):
            return wordnet.ADJ
        elif tag.startswith('V'):
            return wordnet.VERB
        elif tag.startswith('N'):
            return wordnet.NOUN
        elif tag.startswith('R'):
            return wordnet.ADV
        # Por defecto, etiquetar como sustantivo
        else:
            return wordnet.NOUN

    # Tokenizar la oración
    tokens = word_tokenize(sentence)
    # Etiquetar las palabras de acuerdo a sus categorías léxicas
    tagged_tokens = pos_tag(tokens)
    # Instanciar el lematizador
    lemmatizer = WordNetLemmatizer()
    # Lematizar cada palabra considerando su categoría léxica
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged_tokens]
    # Unir las palabras lematizadas en una cadena
    return ' '.join(lemmatized_words)
lemmatize_udf = udf(lemmatize_sentence, StringType())

In [ ]:
# 1. Convertir el texto a minúsculas
# 2. Eliminar saltos de línea
# 3. Eliminar tabulaciones
# 4. Eliminar números y caracteres especiales
# 5. Tokenizar el texto (utilizando los espacios como delimitadores)
# 6. Eliminar palabras cortas (menores a 4 caracteres de longitud) y stopwords
# 7. Reunificar las palabras en una cadena
# 8. Lematizar el texto
processed_comment_df = comment_df \
    .withColumn("lower", lower(col("comment"))) \
    .withColumn("no_newlines", regexp_replace(col("lower"), "\n", " ")) \
    .withColumn("no_tabs", regexp_replace(col("no_newlines"), "\t", " ")) \
    .withColumn("letters_only", regexp_replace(col("no_tabs"), "[^a-zA-Z]", " ")) \
    .withColumn("words", split(col("letters_only"), "\\s+")) \
    .withColumn("filtered_words", filter_words_udf(col("words"))) \
    .withColumn("filtered_text", array_join(col("filtered_words"), " ")) \
    .withColumn("lemmatized_comment", lemmatize_udf(col("filtered_text"))) \
    .select("lemmatized_comment")

In [ ]:
processed_comment_df

In [ ]:
# Streaming de comentarios procesados
processed_comment_df.createOrReplaceTempView("processed_comments")
processed_comment_query = processed_comment_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("processed_comments") \
    .trigger(processingTime="3 minutes") \
    .start()

In [ ]:
import time
while True:
    spark.sql("""
        SELECT *
        FROM processed_comments
        LIMIT 100
    """).show(truncate=False)

    time.sleep(10)  # Espera 10 segundos antes de volver a consultar

In [ ]:
spark.sql("select * from processed_comments limit 5").show()